# Olist: monthly GMV

## Goal
Item-price GMV per month (sum of `price`; freight is kept separate). The main trap: the items table has several rows per order, so it must be aggregated by order before joining to orders. The SQL version is in `sql/02_monthly_gmv.sql`.

## Setup
Load orders and items and rebuild `valid_orders` as in [02-monthly-orders](02-monthly-orders.ipynb).

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Project folder = the nearest parent that contains data/raw
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
DATA_DIR = ROOT / "data" / "raw"

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")

# Valid orders (same definition as 02-monthly-orders)
orders["purchase_ts"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
valid_orders = orders[~orders["order_status"].isin(["canceled", "unavailable"])].copy()
valid_orders["purchase_month"] = valid_orders["purchase_ts"].dt.to_period("M")

print("Valid orders:", len(valid_orders), "| item rows:", len(items))

Valid orders: 98207 | item rows: 112650


## Definitions
- **GMV** = sum of `price` from the items table. `freight_value` is not included.
- **Time** comes from the orders table (`purchase_ts`).
- **Grain**: items are grouped by `order_id` and `price` is summed, giving one `order_gmv` per order.
- **Join**: `valid_orders` left-joined to the per-order amounts, so orders without item rows are kept (their amount is missing and ignored by the sum).
- **Orders**: the same valid orders as the monthly order count.
- **Checks**: `order_id` is still unique after the join; the months add up to the total `price` of the valid orders' items.

In [2]:
# 1. Aggregate items by order: one row per order
order_gmv = (
    items.groupby("order_id", as_index=False)["price"].sum()
    .rename(columns={"price": "order_gmv"})
)
print("Rows after aggregation =", len(order_gmv), "| distinct orders =", order_gmv["order_id"].nunique())

# 2. Left join to valid orders: orders without items are kept
orders_gmv = valid_orders.merge(order_gmv, on="order_id", how="left")

# 3. order_id must still be unique: rows = distinct orders
print("Rows after join =", len(orders_gmv), "| distinct orders =", orders_gmv["order_id"].nunique())
print("Valid orders without item rows =", orders_gmv["order_gmv"].isna().sum())

Rows after aggregation = 98666 | distinct orders = 98666
Rows after join = 98207 | distinct orders = 98207
Valid orders without item rows = 8


In [3]:
# 4. One row per purchase month
monthly_gmv = orders_gmv.groupby("purchase_month")["order_gmv"].sum()
print(monthly_gmv.round(2).to_string())

# 5. Check: the months add up to the item total of valid orders; SQL gives 13494400.74
total_check = items[items["order_id"].isin(valid_orders["order_id"])]["price"].sum()
print("Sum of months:", round(monthly_gmv.sum(), 2))
print("Item total:   ", round(total_check, 2))
print("Equal:", abs(monthly_gmv.sum() - total_check) < 0.01)
print("Months:", len(monthly_gmv))

purchase_month
2016-09        207.86
2016-10      44507.30
2016-12         10.90
2017-01     120098.27
2017-02     244959.35
2017-03     368341.32
2017-04     353842.98
2017-05     503159.19
2017-06     429916.61
2017-07     492287.30
2017-08     568245.79
2017-09     621415.91
2017-10     660179.62
2017-11    1003862.14
2017-12     742183.79
2018-01     945456.29
2018-02     837895.43
2018-03     981051.06
2018-04     993592.98
2018-05     992871.75
2018-06     863265.53
2018-07     878044.27
2018-08     848860.10
2018-09        145.00
Freq: M
Sum of months: 13494400.74
Item total:    13494400.74
Equal: True
Months: 24


## Findings
- Total item-price GMV of valid orders is 13,494,400.74 BRL across 24 months, matching the SQL result.
- The highest month is 2017-11 at 1,003,862.14. From 2018-01 to 2018-08 every month is between 837,895 and 993,593.
- 8 valid orders have no item rows and contribute nothing to GMV.
- GMV here is not company revenue: it excludes freight and does not deduct commissions or refunds.